# Session 11.1: End-to-End Mini-Project (Part 1)
# Project Setup, Data Prep & EDA

<table cellpadding="10" cellspacing="0" border="0" width="100%"><tr>
<td bgcolor="#ED1C24" width="4"></td>
<td bgcolor="#fce4ec">
<font color="#c62828"><b>Program:</b></font> Vishlesan i-Hub IIT Patna x Masai School -- AIM (AI & Machine Learning)<br>
<font color="#c62828"><b>Session ID:</b></font> 11.1 | <b>Week:</b> 11<br>
<font color="#c62828"><b>Prerequisites:</b></font> Session 10.1 (Data Quality Pipelines) · Session 10.2 (EDA Workflow) · Sessions 9.1–9.2 (KNN &amp; Linear Regression)<br>

</tr></table>

---

Vishlesan i-Hub IIT Patna × Masai School

## Learning Objectives

By the end of this notebook you will be able to:

1. **Frame** a vague business brief as a tractable ML problem (target variable, success metric, baseline)
2. **Inspect** a fresh dataset systematically — shape, types, summary stats, missing-value matrix, duplicate audit
3. **Apply** the 6-step cleaning order at industry pace: Duplicates → Strings → Types → Impossibles → Outliers → Missing values (last)
4. **Split** data into train/test *before* EDA to prevent information leakage
5. **Run** a comprehensive EDA workflow on the training set: distributions with KDE overlays, correlation analysis, categorical breakdowns
6. **Engineer** new features (log transforms, derived ratios, frequency encodings) on TRAIN and apply the same transforms to TEST
7. **Compute** a baseline model performance (mean-prediction RMSE) that any future model must beat
8. **Save** cleaned, feature-engineered train and test CSVs as the hand-off to Session 11.2


## Setup & Imports

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

import warnings
warnings.filterwarnings('ignore')
print("All libraries loaded.")

All libraries loaded.


In [2]:
from IPython.display import HTML, display

_BOX_STYLES = {
    "definition": ("#448aff", "#e3f2fd", "#1565c0"),
    "tip":        ("#00c853", "#e8f5e9", "#2e7d32"),
    "warning":    ("#ff9100", "#fff3e0", "#e65100"),
    "danger":     ("#ff1744", "#fce4ec", "#c62828"),
    "math":       ("#7c4dff", "#ede7f6", "#4527a0"),
    "output":     ("#00b8d4", "#e0f7fa", "#006064"),
    "industry":   ("#009688", "#e0f2f1", "#004d40"),
}

def box(kind, title, content):
    border, bg, title_clr = _BOX_STYLES[kind]
    display(HTML(f"""
    <div style="margin:12px 0; padding:12px 16px; border-left:4px solid {border};
                background-color:{bg}; border-radius:4px;">
    <strong style="color:{title_clr};">{title}</strong><br>{content}
    </div>"""))

box("tip", "Setup Complete", "Helpers loaded. Ready to build the GharConnect price-estimation project.")

def kde_line(data, n_points=200):
    """Return (x, y) arrays for a smooth KDE curve."""
    data = data.dropna()
    kde = gaussian_kde(data, bw_method='scott')
    x = np.linspace(data.min(), data.max(), n_points)
    y = kde(x)
    return x, y

## Dataset: Bengaluru Residential Listings

`bengaluru_listings.csv` is a snapshot of ~8,000 residential listings pulled from a Bengaluru property portal, covering 28 neighbourhoods that range from premium core (Indiranagar, Koramangala, Jayanagar) to growing suburbs (Sarjapur Road, Whitefield, Hebbal) to outskirts (Electronic City, Bommasandra, Devanahalli).

Like any aggregator export, it carries the data-quality artefacts you will see on Day 1 at NoBroker, Magicbricks, or 99acres:
- **Re-posted listings** — the same flat appears twice when a different agent re-uploads it or the owner refreshes the price
- **Optional fields left blank** — sellers skip soft attributes like `parking_spots`, sometimes `area_sqft` or `age_years`
- **Locality typed differently by different agents** — mixed case, trailing spaces, and minor variants (`'WHITEFIELD'`, `'white field'`, `' Whitefield '`)
- **Units pasted inline** — some agents type `area_sqft` as `'1200 sqft'` instead of the numeric `1200`
- **Data-entry errors** — `bhk = 0` (typo for a blank), `age_years = -1` (a failed date-difference)
- **Genuine outliers** — luxury penthouses at the top end, studio flats at the bottom

Today's workflow is exactly how the ML team at any of these companies would clean and prepare this export before training a price-estimation model.

In [3]:
import os

# Load the dataset
# In Google Colab: upload bengaluru_listings.csv first, or mount your Drive
df = pd.read_csv('bengaluru_listings.csv')

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)

Dataset loaded: 8240 rows × 9 columns


,listing_id,locality,bhk,area_sqft,bathrooms,age_years,furnishing_status,parking_spots,price_lakh
0,11920,Devanahalli,4,2283.7,3.0,11.3,Unfurnished,NaN,111.38
1,14656,Electronic City,2,1017.4,2.0,7.9,Semi-Furnished,1.0,59.75
2,11840,Hennur,4,2179.7,3.0,19.4,Unfurnished,3.0,140.14


---

## Business Context: GharConnect PropTech


In [4]:
box("industry", "Indian PropTech: ML Is Already Here",
    "NoBroker (India's first PropTech unicorn) uses ML for <b>fraud detection</b> on listings — "
    "fake floor plans, bait-and-switch prices, ghost listings. "
    "Magicbricks has faced public criticism for <b>biased pricing models</b> in Tier-II areas. "
    "99acres uses ML for <b>owner verification</b> to reduce broker fraud. "
    "Housing.com's valuation engine powers <b>bank mortgage approvals</b> in India. "
    "<b>The Indian PropTech ML market is ~₹2,400 Cr and growing at 28% CAGR.</b> "
    "A junior ML engineer at any of these companies would start their first project "
    "with exactly what we are doing today.")

In [5]:
box("industry", "The CEO Email — GharConnect",
    "<i>You are three months into your first job as a junior ML engineer at <b>GharConnect</b>, "
    "a Bengaluru PropTech startup. On Monday morning the CEO drops a CSV from the listings portal "
    "into your inbox — roughly 8,000 rows of recent residential listings — with this message:</i><br><br>"
    "<blockquote style='border-left:3px solid #009688; margin-left:16px; padding-left:12px;'>"
    "<i>'Hey — can you build a price-estimation model for our listings? "
    "Something to help first-time homebuyers avoid getting overcharged. "
    "Got it back to me by Friday. Thanks.'</i>"
    "</blockquote>"
    "<b>That is the entire brief.</b> No target column. No success metric. No mention of which model. "
    "This is what 90% of real ML work looks like. The meta-skill — converting a vague brief into a "
    "tractable ML problem — is what we build today.")

> **Analogy: The Architect's Blueprint.** A contractor who starts hammering bricks without a blueprint will build the wrong building. A problem statement document is the ML engineer's blueprint — it defines *what* you are building before you write a single line of modelling code.


---

## Section 1: Problem Framing — Converting a Vague Brief into an ML Problem

**This is the most underrated skill in ML practice.** Below is the 5-step problem framing process every junior engineer needs. It takes 10 minutes. It saves 10 days.

```
           ┌─ Vague brief from CEO ─┐
           │  'Help homebuyers       │
           │   avoid overpaying'    │
           └─────────┬──────────────┘
                     │
         ┌───────────┴───────────┐
         │                       │
    What can we              Regression or
    PREDICT from             Classification?
    this data?                     │
         │                         ▼
         ▼                  Predict a continuous
    price_lakh             price → Regression
    (₹ in lakhs)
         │
         ▼
    Target   = price_lakh
    Metric   = RMSE (₹ lakhs)
    Baseline = predict mean price for every listing
    Success  = beat baseline RMSE by 30%+
```


In [6]:
box("definition", "5-Step Problem Framing Process",
    "<b>Step 1: What is the BUSINESS question?</b><br>"
    "Translate the CEO's goal into a measurable outcome. "
    "Here: 'help homebuyers avoid overpaying' = 'give buyers an estimate of fair market price'.<br><br>"
    "<b>Step 2: What is the ML question?</b><br>"
    "Can we predict the business outcome from available features? "
    "Here: Yes — listing features (area, locality, BHK, age, furnishing) predict price.<br><br>"
    "<b>Step 3: What is the TARGET variable?</b><br>"
    "The column we want to predict. Here: <code>price_lakh</code> (continuous → regression).<br><br>"
    "<b>Step 4: What is the SUCCESS METRIC?</b><br>"
    "RMSE (Root Mean Squared Error) in ₹ lakhs. Lower is better. Units are interpretable.<br><br>"
    "<b>Step 5: What is the BASELINE?</b><br>"
    "Predict the mean training price for every test row. Any real model must beat this. "
    "If it cannot — the model has zero signal.")

### Step 1: Define the Problem Statement Programmatically

In [7]:
# A problem statement document is version-controlled just like code.
# This discipline separates engineers from script-kiddies.

PROBLEM_STATEMENT = {
    "business_question":  "Help first-time homebuyers in Bengaluru avoid overpaying for residential listings",
    "ml_question":        "Predict listing price (in Rs. lakhs) from listing features",
    "target_variable":    "price_lakh",
    "success_metric":     "RMSE (lower is better; units = Rs. lakhs)",
    "baseline_strategy":  "Predict mean training price for every test listing",
    "success_threshold":  "Beat baseline RMSE by at least 30%",
}

print("PROBLEM STATEMENT")
print("=" * 60)
for k, v in PROBLEM_STATEMENT.items():
    print(f"{k:<22} : {v}")

PROBLEM STATEMENT
business_question      : Help first-time homebuyers in Bengaluru avoid overpaying for residential listings
ml_question            : Predict listing price (in Rs. lakhs) from listing features
target_variable        : price_lakh
success_metric         : RMSE (lower is better; units = Rs. lakhs)
baseline_strategy      : Predict mean training price for every test listing
success_threshold      : Beat baseline RMSE by at least 30%


### Step 2: The Data Dictionary

In [8]:
box("definition", "What Is a Data Dictionary?",
    "A data dictionary documents every column in the dataset before you write any cleaning code. "
    "It records: column name, data type, description, allowed range, and a sample value. "
    "<b>This document prevents 80% of cleaning mistakes</b> — because you know what 'valid' looks "
    "like before you start filtering. Teams also use it to onboard new engineers within minutes.")

| Column | Type | Description | Allowed Range | Sample |
|--------|------|-------------|---------------|--------|
| `listing_id` | int | Unique listing identifier | 10001–18000 | 10042 |
| `locality` | str | Bengaluru neighbourhood | 28 canonical names | Koramangala |
| `bhk` | int | Bedrooms + hall + kitchen | 1–5 | 2 |
| `area_sqft` | float | Total built-up area (sq ft) | 250–4500 | 1150.5 |
| `bathrooms` | int | Number of bathrooms | 1–5 | 2 |
| `age_years` | float | Property age in years | 0–30 | 4.3 |
| `furnishing_status` | str | Furnishing level | Furnished / Semi-Furnished / Unfurnished | Semi-Furnished |
| `parking_spots` | int | Covered parking slots | 0–3 | 1 |
| `price_lakh` | float | Listing price in ₹ lakhs (TARGET) | ~20–700 | 87.5 |


In [9]:
# Build data dictionary as a DataFrame for easy reference
data_dict = pd.DataFrame({
    'Column':        ['listing_id', 'locality', 'bhk', 'area_sqft',
                      'bathrooms', 'age_years', 'furnishing_status', 'parking_spots', 'price_lakh'],
    'Type':          ['int', 'str', 'int', 'float',
                      'int', 'float', 'str', 'int', 'float'],
    'Description':   ['Unique listing ID', 'Bengaluru neighbourhood',
                      'Bedrooms + hall + kitchen', 'Built-up area (sq ft)',
                      'Number of bathrooms', 'Property age (years)',
                      'Furnishing level', 'Covered parking slots',
                      'Price in Rs. lakhs (TARGET)'],
    'Allowed_Range': ['10001-18000', '28 canonical names', '1-5', '250-4500',
                      '1-5', '0-30', 'Furnished/Semi-Furnished/Unfurnished',
                      '0-3', '~20-700'],
    'Sample':        [10042, 'Koramangala', 2, 1150.5, 2, 4.3, 'Semi-Furnished', 1, 87.5]
})

print("DATA DICTIONARY")
display(data_dict)

DATA DICTIONARY


,Column,Type,Description,Allowed_Range,Sample
0,listing_id,int,Unique listing ID,10001-18000,10042
1,locality,str,Bengaluru neighbourhood,28 canonical names,Koramangala
2,bhk,int,Bedrooms + hall + kitchen,1-5,2
3,area_sqft,float,Built-up area (sq ft),250-4500,1150.5
4,bathrooms,int,Number of bathrooms,1-5,2
5,age_years,float,Property age (years),0-30,4.3
6,furnishing_status,str,Furnishing level,Furnished/Semi-Furnished/Unfurnished,Semi-Furnished
7,parking_spots,int,Covered parking slots,0-3,1
8,price_lakh,float,Price in Rs. lakhs (TARGET),~20-700,87.5


In [10]:
box("definition", "What Is a Baseline Model?",
    "A baseline is the <b>dumbest possible prediction</b>. For regression, the baseline predicts "
    "the mean of the training target for <i>every</i> test row — regardless of any features. "
    "<b>Why?</b> Because if your sophisticated model cannot beat a trivial mean-prediction, "
    "it has zero signal and should not be deployed. "
    "The baseline RMSE is the <b>floor</b>. Every model you train in 11.2 must beat it.<br><br>"
    "Baseline prediction: <code>price_hat = mean(train price_lakh)</code> for every test row.")

---

## Section 2: Initial Data Inspection

The 5-thing first-look audit — run this on every fresh dataset:

| Audit | Function | What to look for |
|-------|----------|------------------|
| Shape | `df.shape` | Row count matches expectations? |
| Types | `df.dtypes` | Any column the wrong type? |
| Summary | `df.describe(include='all')` | Min/max sane? Unusual std? |
| Missing | `df.isna().sum()` | Which columns? What %, what pattern? |
| Duplicates | `df.duplicated().sum()` | How many exact duplicates? |


In [11]:
# --- Audit 1 & 2: Shape and Types ---
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nFirst 3 rows:")
display(df.head(3))

Shape: 8240 rows x 9 columns

Column types:
listing_id             int64
locality              object
bhk                    int64
area_sqft             object
bathrooms            float64
age_years            float64
furnishing_status     object
parking_spots        float64
price_lakh           float64
dtype: object

First 3 rows:


,listing_id,locality,bhk,area_sqft,bathrooms,age_years,furnishing_status,parking_spots,price_lakh
0,11920,Devanahalli,4,2283.7,3.0,11.3,Unfurnished,NaN,111.38
1,14656,Electronic City,2,1017.4,2.0,7.9,Semi-Furnished,1.0,59.75
2,11840,Hennur,4,2179.7,3.0,19.4,Unfurnished,3.0,140.14


In [12]:
# --- Audit 3: Statistical Summary ---
# Flag any column whose count < total rows (indicates missing values)
display(df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
listing_id,8240.0,NaN,NaN,NaN,13993.937743,2310.346111,10001.0,11989.75,13988.5,15994.25,18000.0
locality,8240,107,Whitefield,580,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bhk,8240.0,NaN,NaN,NaN,2.562864,0.895935,0.0,2.0,3.0,3.0,5.0
area_sqft,7583,5955,1006.9,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bathrooms,7993.0,NaN,NaN,NaN,2.252346,0.949423,1.0,2.0,2.0,3.0,5.0
age_years,7746.0,NaN,NaN,NaN,12.701782,7.517984,-1.0,6.3,12.7,18.9,30.0
furnishing_status,8240,12,Semi-Furnished,3337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
parking_spots,7252.0,NaN,NaN,NaN,1.488417,0.836192,0.0,1.0,1.0,2.0,3.0
price_lakh,8240.0,NaN,NaN,NaN,111.910808,70.034101,12.0,68.6,96.275,136.615,698.83


### Reading the `describe()` Output — Spotting Anomalies

`df.describe()` is not a passive table — it is an *audit checklist*. For each numeric column, look at:

- **`count`** — if it's less than the total row count, that column has missing values
- **`min` / `max`** — do they make physical sense? (`age_years.min() = -1` is a red flag; a building cannot be from the future)
- **`std`** vs **`mean`** — a std that is a large fraction of the mean hints at outliers or a long tail
- **`50%` (median)** vs **`mean`** — a big gap signals skew

For categorical / string columns, `describe(include='all')` shows **`unique`** and **`top`** — if `unique` is much larger than the number of real categories, you have casing / whitespace inconsistencies.

Let us now run targeted checks on each column the describe output flagged.

In [13]:
# Anomaly hunt — drill into the columns flagged by describe()

# Discrete-numeric column: full value distribution
print('bhk value counts:')
print(df['bhk'].value_counts().sort_index().to_string())
print()

# Continuous numeric: explicit anomaly counts
n_age_neg     = (df['age_years'] < 0).sum()
print(f'Rows with age_years < 0     : {n_age_neg}')

# Type audit on a column that should be numeric
print(f"area_sqft dtype             : {df['area_sqft'].dtype}")
type_counts = df['area_sqft'].apply(type).value_counts()
print(f'area_sqft value types       :')
for t, c in type_counts.items():
    print(f'   {t.__name__:<8} : {c}')
print(f'Sample non-numeric area values: {df[df["area_sqft"].apply(lambda x: isinstance(x, str))]["area_sqft"].head(3).tolist()}')
print()

# Categorical / string audit: too many unique values for the real category count
n_unique_loc = df['locality'].nunique()
print(f'Unique locality strings     : {n_unique_loc}  (we expect ~28 real neighbourhoods)')
print('Sample locality variants (case/whitespace mismatches):')
print(df['locality'].value_counts().head(15).to_string())

bhk value counts:
bhk
0      24
1     782
2    3253
3    3046
4     983
5     152

Rows with age_years < 0     : 12
area_sqft dtype             : object
area_sqft value types       :
   str      : 7583
   float    : 657
Sample non-numeric area values: ['2283.7', '1017.4', '2179.7']

Unique locality strings     : 107  (we expect ~28 real neighbourhoods)
Sample locality variants (case/whitespace mismatches):
locality
Whitefield           580
Sarjapur Road        476
Koramangala          430
HSR Layout           410
Indiranagar          395
Electronic City      379
Jayanagar            365
Marathahalli         348
Bellandur            339
Banashankari         319
Bannerghatta Road    300
Yelahanka            296
BTM Layout           287
Hebbal               277
JP Nagar             269


In [14]:
# Pull the anomaly counts out as named variables so the interpretation
# below is computed, never hardcoded.
n_bhk_zero    = (df['bhk'] == 0).sum()
n_age_neg     = (df['age_years'] < 0).sum()
n_area_str    = df['area_sqft'].apply(lambda x: isinstance(x, str)).sum()
n_loc_unique  = df['locality'].nunique()

box("output", "Anomalies Discovered in the describe() Output",
    f"<b>Impossible values found:</b><br>"
    f"&nbsp;&nbsp;• <code>bhk = 0</code> in <b>{n_bhk_zero}</b> rows — a flat cannot have zero bedrooms-hall-kitchen<br>"
    f"&nbsp;&nbsp;• <code>age_years &lt; 0</code> in <b>{n_age_neg}</b> rows — a building cannot be from the future<br><br>"
    f"<b>Type issue found:</b><br>"
    f"&nbsp;&nbsp;• <code>area_sqft</code> dtype is <code>object</code> with <b>{n_area_str}</b> rows storing values like <code>'1200 sqft'</code><br><br>"
    f"<b>String inconsistencies found:</b><br>"
    f"&nbsp;&nbsp;• <code>locality</code> has <b>{n_loc_unique}</b> distinct strings for ~28 real neighbourhoods — "
    f"different agents typed the same place differently (case, whitespace, capitalisation)<br><br>"
    f"<b>Outliers (suspected):</b> <code>price_lakh</code> has a wide range — we will visualise this in Step 5 of cleaning before deciding how to handle it.")

### Missing Value Analysis

Look for columns with **count < total rows** in the summary above — those have missing values. The bar chart below shows exactly how many and what percentage.

In [15]:
# --- Audit 4: Missing Value Bar Chart ---
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
miss_df = pd.DataFrame({'Missing Count': missing, 'Percent': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Percent', ascending=True)

fig = go.Figure(go.Bar(
    y=miss_df.index,
    x=miss_df['Percent'],
    orientation='h',
    marker_color='#ED1C24',
    text=[f"{int(c)} ({p}%)" for c, p in zip(miss_df['Missing Count'], miss_df['Percent'])],
    textposition='auto',
    hovertemplate='%{y}: %{x:.1f}% missing<extra></extra>'
))
fig.update_layout(
    title='Missing Values per Column (% of total rows)',
    xaxis_title='Missing %',
    width=700, height=350,
    template='simple_white'
)
fig.show()

In [16]:
n_missing_total = df.isna().sum().sum()
n_rows_with_missing = df.isna().any(axis=1).sum()
worst_col = df.isna().sum().idxmax()
worst_pct = df.isna().sum().max() / len(df) * 100

box("output", "Missing Value Audit",
    f"Total missing cells: <b>{n_missing_total:,}</b> across {df.shape[1]} columns.<br>"
    f"Rows with at least one missing value: <b>{n_rows_with_missing:,}</b> ({n_rows_with_missing/len(df)*100:.1f}%).<br>"
    f"Worst column: <b>{worst_col}</b> ({worst_pct:.1f}% missing).<br>"
    f"<b>Plan:</b> Missing value imputation comes LAST in the cleaning order — "
    f"after duplicates, strings, types, impossibles, and outliers are handled.")

In [17]:
# --- Audit 5: Duplicates ---
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dupes}")
if n_dupes > 0:
    print(f"Example duplicate (first 2 copies of first duplicated row):")
    display(df[df.duplicated(keep=False)].head(2)[['listing_id', 'locality', 'bhk', 'area_sqft', 'price_lakh']])

Exact duplicate rows: 101
Example duplicate (first 2 copies of first duplicated row):


,listing_id,locality,bhk,area_sqft,price_lakh
25,11438,Whitefield,3,1624.6,102.71
63,14053,Whitefield,1,829.5,63.45


### Cleaning Plan — From Discoveries to Actions

Our 5-thing audit + describe-driven anomaly hunt + missing-value chart surfaced the following six issues. Each one drives a specific cleaning step, and the **order matters** (we explain why on the next cell).

| # | Issue (discovered) | Cleaning step | Technique |
|---|--------------------|---------------|-----------|
| 1 | Exact duplicate rows from re-postings | Step 1 | `drop_duplicates()` |
| 2 | `locality` has many case / whitespace variants | Step 2 | `.str.strip().str.title()` + acronym fix-map |
| 3 | `area_sqft` dtype `object` (mixed `'1200 sqft'` strings) | Step 3 | custom `to_float_sqft()` parser |
| 4 | Impossible values (`bhk = 0`, `age_years < 0`) | Step 4 | row filter (preserve NaN for Step 6) |
| 5 | Suspected price outliers (long right tail) | Step 5 | percentile-cap (winsorize) |
| 6 | Missing values in 4 columns | Step 6 | strategy chosen per column (median / mode / 0) |


In [18]:
area_dtype = df['area_sqft'].dtype
area_obj_count = (df['area_sqft'].apply(lambda x: isinstance(x, str))).sum()
bhk_zero_count = (pd.to_numeric(df['bhk'], errors='coerce') == 0).sum()
age_neg_count  = (pd.to_numeric(df['age_years'], errors='coerce') < 0).sum()

box("warning", "Cleaning Plan — 6 Issues Detected",
    f"<b>Step 1 — Duplicates:</b> {df.duplicated().sum()} exact duplicate rows → drop_duplicates()<br>"
    f"<b>Step 2 — Strings:</b> locality mixed case/spaces → .str.strip().str.title()<br>"
    f"<b>Step 3 — Types:</b> area_sqft is {area_dtype} (has {area_obj_count} string values) → to_float_sqft()<br>"
    f"<b>Step 4 — Impossibles:</b> {bhk_zero_count} bhk=0 rows, {age_neg_count} age<0 rows → filter out<br>"
    f"<b>Step 5 — Outliers:</b> price_lakh extremes → 1st–99th percentile cap<br>"
    f"<b>Step 6 — Missing (LAST):</b> area_sqft/age_years/bathrooms→median, parking_spots→0, furnishing→mode → fillna()")

---

## Section 3: Data Cleaning — The 6-Step Order

> **Why order matters:** You cannot impute missing values BEFORE removing impossible values — because the impossibles pollute your imputation statistics (e.g., `age_years = -1` would pull down the median used to fill NaNs). Always follow: Duplicates → Strings → Types → Impossibles → Outliers → Missing (last).


### Step 1: Remove Exact Duplicates

In [19]:
before_rows = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
after_rows = len(df)

print(f"Duplicate rows removed: {before_rows - after_rows}")
print(f"Rows: {before_rows} --> {after_rows}")

Duplicate rows removed: 101
Rows: 8240 --> 8139


### Step 2: String Standardisation

Our discovery cell showed the `locality` column has many more distinct strings than there are real neighbourhoods. The first tool to reach for is `.str.strip().str.title()` — it strips leading/trailing whitespace and applies title-case in a single chained call. Watch what happens, then look carefully at the result.

In [20]:
# Stage A: apply .str.strip().str.title() and observe the result
n_before = df['locality'].nunique()
df['locality'] = df['localita
n_after_stage_a = df['locality'].nunique()

print(f"Unique localities — BEFORE strip+title : {n_before}")
print(f"Unique localities — AFTER  strip+title : {n_after_stage_a}")
print()
print('Top 20 localities now (sorted by count):')
print(df['locality'].value_counts().head(20).to_string())
print()
print('Bottom 8 localities (sorted by count):')
print(df['locality'].value_counts().tail(8).to_string())

Unique localities — BEFORE strip+title : 107
Unique localities — AFTER  strip+title : 29

Top 20 localities now (sorted by count):
locality
Whitefield           595
Sarjapur Road        490
Koramangala          457
Hsr Layout           432
Indiranagar          425
Electronic City      397
Jayanagar            381
Marathahalli         368
Bellandur            359
Banashankari         331
Bannerghatta Road    314
Yelahanka            307
Btm Layout           305
Hebbal               297
Jp Nagar             284
Kr Puram             250
Thanisandra          249
Kanakapura Road      237
Hoodi                221
Hennur               214

Bottom 8 localities (sorted by count):
locality
Cv Raman Nagar    180
Mysore Road       176
Bommasandra       164
Begur             150
Devanahalli       141
Attibele          126
Sadashivnagar      90
White Field         8


**Look at the bottom of that list carefully.** Do you see entries like `Hsr Layout`, `Btm Layout`, `Jp Nagar`, `Kr Puram`, `Cv Raman Nagar`?

Those are the **same neighbourhoods** as `HSR Layout`, `BTM Layout`, etc. — but `.str.title()` capitalised only the first letter of each word, so it turned the all-caps acronyms `HSR`, `BTM`, `JP`, `KR`, `CV` into `Hsr`, `Btm`, `Jp`, `Kr`, `Cv`.

Same for `White Field` (was `'white field'`) — it should be `Whitefield` (one word).

**This is a classic gotcha** with Indian locality names because so many of them use acronym prefixes. `.str.title()` is not wrong — it is too aggressive. The fix is a small lookup table that maps each mis-cased variant back to the canonical form.

In [21]:
# Stage B: small correction map for multi-word + acronym localities
# Each entry maps the mangled .str.title() output back to the canonical name.
locality_fix_map = {
    'White Field':    'Whitefield',     # multi-word -> single word
    'Hsr Layout':     'HSR Layout',     # acronym restored
    'Btm Layout':     'BTM Layout',     # acronym restored
    'Jp Nagar':       'JP Nagar',       # acronym restored
    'Kr Puram':       'KR Puram',       # acronym restored
    'Cv Raman Nagar': 'CV Raman Nagar', # acronym restored
}
df['locality'] = df['locality'].replace(locality_fix_map)

# Apply the same strip+title to furnishing_status (no acronym issues here,
# so no fix-map needed)
df['furnishing_status'] = df['furnishing_status'].str.strip().str.title()

n_after_stage_b = df['locality'].nunique()
print(f"Unique localities — AFTER fix-map        : {n_after_stage_b}")
print(f"Reduction across both stages             : {n_before} → {n_after_stage_a} → {n_after_stage_b}")
print()
print('Final canonical localities (counts):')
print(df['locality'].value_counts().to_string())

Unique localities — AFTER fix-map        : 28
Reduction across both stages             : 107 → 29 → 28

Final canonical localities (counts):
locality
Whitefield           603
Sarjapur Road        490
Koramangala          457
HSR Layout           432
Indiranagar          425
Electronic City      397
Jayanagar            381
Marathahalli         368
Bellandur            359
Banashankari         331
Bannerghatta Road    314
Yelahanka            307
BTM Layout           305
Hebbal               297
JP Nagar             284
KR Puram             250
Thanisandra          249
Kanakapura Road      237
Hoodi                221
Hennur               214
Domlur               191
CV Raman Nagar       180
Mysore Road          176
Bommasandra          164
Begur                150
Devanahalli          141
Attibele             126
Sadashivnagar         90


### Step 3: Type Conversion

`area_sqft` has ~3% of values stored as strings like `'1200 sqft'` or `'1,200 sqft'`. We strip the unit and commas, then convert to `float`. Rows that are already `float` (or `NaN`) are returned unchanged.

In [22]:
def to_float_sqft(x):
    """Convert area values like '1200 sqft' or '1,200 sqft' to float."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        cleaned = x.replace('sqft', '').replace(',', '').strip()
        try:
            return float(cleaned)
        except ValueError:
            return np.nan
    return float(x)

df['area_sqft'] = df['area_sqft'].apply(to_float_sqft)
print(f"area_sqft dtype now: {df['area_sqft'].dtype}")
print(f"Sample values: {df['area_sqft'].dropna().head(5).round(1).tolist()}")

area_sqft dtype now: float64
Sample values: [2283.7, 1017.4, 2179.7, 1293.2, 1805.8]


### Step 4: Remove Impossible Values

In [23]:
box("definition", "Impossible vs Outlier vs Missing — Three Different Problems",
    "<b>Impossible value</b> — a value that <i>cannot exist</i> by physical / logical / domain rules. "
    "Examples: a flat with <code>bhk = 0</code>, a building with <code>age_years &lt; 0</code>, "
    "an area of <code>0 sqft</code>. These are <b>data-entry errors or pipeline bugs</b> — "
    "the value is wrong, not extreme. <b>Action:</b> remove the row.<br><br>"
    "<b>Outlier</b> — a value that is <i>extreme but legitimate</i>. "
    "Example: a ₹6 Cr penthouse in Indiranagar — rare, real, expensive. "
    "<b>Action:</b> cap, transform, flag, or sometimes remove (we cover the trade-offs in Step 5).<br><br>"
    "<b>Missing value (NaN)</b> — the value was simply not recorded. "
    "Different from impossible (a wrong value was recorded) and outlier (an extreme value was recorded). "
    "<b>Action:</b> impute in Step 6, after we know what 'clean' looks like.<br><br>"
    "<b>Why this matters now:</b> Step 4 must remove only the <i>impossibles</i>, while leaving NaNs alone "
    "so Step 6 can impute them properly.")

**Recap of the impossibles we found in our discovery cell:** `bhk = 0` rows and `age_years < 0` rows. Let's confirm the counts one more time on the *current* state of `df` (post-string-cleaning, post-type-conversion) before we filter them out.

In [24]:
# Re-check impossibles after string-cleaning and type-conversion — these counts
# are what we will actually remove in the next cell.
n_bhk_zero_now = (df['bhk'] == 0).sum()
n_age_neg_now  = (df['age_years'] < 0).sum()
n_area_lt100   = (df['area_sqft'] < 100).sum()  # NaN compares False -> not counted, which is correct
print(f"Rows with bhk = 0          : {n_bhk_zero_now}")
print(f"Rows with age_years < 0    : {n_age_neg_now}")
print(f"Rows with area_sqft < 100  : {n_area_lt100}")
print(f"Total impossible rows to drop: {n_bhk_zero_now + n_age_neg_now + n_area_lt100} (some may overlap)")

Rows with bhk = 0          : 24
Rows with age_years < 0    : 12
Rows with area_sqft < 100  : 0
Total impossible rows to drop: 36 (some may overlap)


In [25]:
before = len(df)

# Filter only the EXPLICITLY impossible rows. Preserve NaN values — those
# are a separate problem and will be handled by imputation in Step 6.
# (Without `.isna() |` the comparisons return False on NaN and drop rows
# that should be imputed.)
df = df[df['bhk'] > 0]                                             # bhk=0 is impossible
df = df[df['age_years'].isna() | (df['age_years'] >= 0)]           # age<0 is impossible; keep NaN for imputation
df = df[df['area_sqft'].isna() | (df['area_sqft'] >= 100)]         # area<100 sqft impossible; keep NaN for imputation
df = df.reset_index(drop=True)

after = len(df)
print(f"Impossible rows removed: {before - after}")
print(f"Rows: {before} --> {after}")

Impossible rows removed: 35
Rows: 8139 --> 8104


### Step 5: Outlier Handling — Detect First, Then Decide

In [26]:
box("definition", "What Is an Outlier? Three Common Detection Rules",
    "An <b>outlier</b> is a value that lies far from the bulk of the distribution. Unlike an impossible "
    "value, it is real — it just sits in the tail. Three standard ways to define 'far':<br><br>"
    "<b>1. IQR rule (Tukey, 1977)</b> — anything below <code>Q1 - 1.5 × IQR</code> or above <code>Q3 + 1.5 × IQR</code> "
    "is an outlier. Used by the box-plot whiskers. Robust because it depends on quartiles, not the mean.<br>"
    "<b>2. Percentile rule</b> — anything below the 1st percentile or above the 99th percentile is an outlier. "
    "Simpler, more aggressive than IQR, and easy to tune (use 5th/95th for a softer cut).<br>"
    "<b>3. Z-score rule</b> — anything more than ~3 standard deviations from the mean. "
    "Works well for symmetric / Gaussian-like distributions — but the mean and std are themselves pulled by "
    "the very outliers we are trying to find, so this rule is fragile on skewed data.<br><br>"
    "<b>Analogy.</b> Imagine ranking flats by price. The IQR rule says 'anything more than one-and-a-half "
    "middle-50% widths beyond the middle is unusual'. The percentile rule says 'the bottom 1% and top 1% "
    "are outliers, period'. Both are valid — choice depends on how aggressive you want to be.")

#### Detect: visualise `price_lakh` to see whether outliers exist

**How to read a boxplot:** the box covers the middle 50% of values (Q1 to Q3). The line in the middle is the median. The whiskers extend to the most extreme values that are *within* `Q3 + 1.5×IQR` (or `Q1 - 1.5×IQR`). Anything beyond the whiskers — shown as individual points — is an IQR outlier.

In [27]:
# Boxplot of price_lakh — the standard first chart for outlier detection
fig = go.Figure(go.Box(
    x=df['price_lakh'],
    name='price_lakh',
    marker_color='#ED1C24',
    boxpoints='outliers',          # only show points beyond the whiskers
    hovertemplate='Price: Rs.%{x:.1f}L<extra></extra>',
))
fig.update_layout(
    title='Price Distribution with Outliers Highlighted (whole dataset, pre-cap)',
    xaxis_title='Price (Rs. Lakhs)',
    width=900, height=300,
    template='simple_white',
    showlegend=False,
)
fig.show()

In [28]:
# Quantitative outlier audit: percentile and IQR bounds, plus counts beyond each
q1, q3 = df['price_lakh'].quantile([0.25, 0.75])
iqr = q3 - q1
iqr_low, iqr_high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
p01, p99 = df['price_lakh'].quantile([0.01, 0.99])

n_iqr_out  = ((df['price_lakh'] < iqr_low) | (df['price_lakh'] > iqr_high)).sum()
n_pct_out  = ((df['price_lakh'] < p01) | (df['price_lakh'] > p99)).sum()
n_ge_500   = (df['price_lakh'] >= 500).sum()

box("output", "Outlier Audit on price_lakh",
    f"<b>IQR rule</b> — flags any value outside [Rs.{iqr_low:.1f}L, Rs.{iqr_high:.1f}L] "
    f"(Q1=Rs.{q1:.1f}L, Q3=Rs.{q3:.1f}L, IQR=Rs.{iqr:.1f}L). "
    f"<b>{n_iqr_out}</b> rows flagged as IQR outliers.<br><br>"
    f"<b>1st–99th percentile rule</b> — flags any value outside [Rs.{p01:.1f}L, Rs.{p99:.1f}L]. "
    f"<b>{n_pct_out}</b> rows flagged.<br><br>"
    f"<b>Eye-test</b> — <b>{n_ge_500}</b> listings ≥ Rs.500L (real luxury flats — not data errors). "
    f"The right tail is real; the question is what to do with it.")

In [71]:
box("definition", "Four Outlier-Handling Strategies — When to Use Each",
    "<b>1. REMOVE</b> — drop the outlier rows. <b>Use when</b> outliers are clearly errors (you have already "
    "separated impossibles from real outliers — for the latter, removing means throwing away real data). <br>"
    "<b>2. CAP / WINSORIZE</b> — clip values to a chosen percentile (e.g., 1st–99th). The row stays; only the "
    "value is bounded. <b>Use when</b> you want to keep every row but stop extreme values from distorting "
    "model training. Standard for regression on right-tailed targets like price.<br>"
    "<b>3. TRANSFORM</b> — apply <code>log1p</code>, square root, or Box-Cox. The values are not changed; the "
    "<i>scale</i> is. <b>Use when</b> the distribution is skewed and the model is sensitive to scale. "
    "We will do this <i>in addition</i> for <code>price_lakh</code> in the Feature Engineering section.<br>"
    "<b>4. FLAG</b> — add a binary indicator column <code>is_outlier</code>. <b>Use when</b> outliers are "
    "informative (e.g., fraud-detection: outlier transactions are exactly what you want to flag).<br><br>"
    "<b>Choice for this project: percentile capping (1st–99th).</b> "
    "Luxury flats at Rs.6Cr+ are real listings — REMOVE would discard them. FLAG would require a "
    "downstream model that uses the flag, which Linear Regression / KNN do not. TRANSFORM (log) we will "
    "apply later as well — but cap-then-log is a common combination because the cap stops the log from "
    "being dominated by a handful of extreme values.")

In [30]:
# Apply: percentile capping (winsorization) on price_lakh
# .clip() keeps every row — values below q_low become q_low, above q_high become q_high.
q_low  = df['price_lakh'].quantile(0.01)
q_high = df['price_lakh'].quantile(0.99)

n_clipped_low  = (df['price_lakh'] < q_low).sum()
n_clipped_high = (df['price_lakh'] > q_high).sum()

df['price_lakh'] = df['price_lakh'].clip(lower=q_low, upper=q_high)

print(f"1st percentile  = Rs.{q_low:.1f}L  (low cap)")
print(f"99th percentile = Rs.{q_high:.1f}L  (high cap)")
print(f"\nValues capped at lower bound : {n_clipped_low}")
print(f"Values capped at upper bound : {n_clipped_high}")
print(f"Rows preserved (no row dropped): {len(df)}")

1st percentile  = Rs.23.8L  (low cap)
99th percentile = Rs.344.5L  (high cap)

Values capped at lower bound : 82
Values capped at upper bound : 82
Rows preserved (no row dropped): 8104


### Step 6: Missing Value Imputation — LAST

**Order matters:** Now that duplicates, impossible values, and outliers are removed, the median and mode statistics are computed on clean data only. If we had imputed first, the `age_years = -1` rows would have dragged down the median, and the Rs.700L outliers would have pulled `area_sqft.median()` away from the typical flat.

In [31]:
box("definition", "Choosing an Imputation Strategy — Six Options",
    "<b>1. Mean</b> — average of non-missing values. <b>Use when:</b> column is roughly symmetric / Gaussian. "
    "<b>Risk:</b> sensitive to skew and remaining outliers — pulled toward the long tail.<br>"
    "<b>2. Median</b> — middle value. <b>Use when:</b> column is numeric and skewed, or when robustness matters. "
    "<b>Strength:</b> not pulled by extreme values, even if a few survived earlier steps.<br>"
    "<b>3. Mode</b> — most frequent value. <b>Use when:</b> column is categorical (strings, labels). "
    "There is no 'mean' or 'median' for non-numeric data.<br>"
    "<b>4. Domain default (e.g., 0)</b> — a constant chosen because it is meaningful in the domain. "
    "<b>Use when:</b> 'missing' has a known semantic (e.g., a real-estate listing with no <code>parking_spots</code> "
    "entered usually means 'no covered parking documented', not 'unknown' — so 0 captures the truth better than the median).<br>"
    "<b>5. KNN-impute</b> — fill the gap with values from the k nearest similar rows. "
    "<b>Use when:</b> rows have many features and similarity is well-defined. More accurate but more compute.<br>"
    "<b>6. Model-based</b> — train a small model (regression / random forest) to predict the missing column from the others. "
    "<b>Use when:</b> the missing column matters a lot and you have time. Powerful but adds complexity.<br><br>"
    "For Day-1 cleaning we use options 2 / 3 / 4. KNN-impute and model-based imputation are tools to reach for "
    "only if the simple options leave too much variance unexplained.")

**Why median / mode / 0 — let the data justify the choice.** A right-skewed numeric column should use median (mean is unreliable). A roughly symmetric numeric column can use mean OR median (median is the safer default). A categorical column needs mode.

Let us check the **skewness** of the numeric columns we need to impute, so the strategy is justified by the data:

In [32]:
# Skewness check — for numeric columns we are about to impute
skews = {
    'area_sqft':     df['area_sqft'].skew(),
    'age_years':     df['age_years'].skew(),
    'bathrooms':     df['bathrooms'].skew(),
    'parking_spots': df['parking_spots'].skew(),
}
print('Skewness of imputation candidates:')
for col, sk in skews.items():
    shape = ('symmetric' if abs(sk) < 0.5 else
             'mildly skewed' if abs(sk) < 1.0 else
             'strongly skewed')
    direction = 'right' if sk > 0 else 'left'
    print(f"  {col:<14}: skew = {sk:+.2f}  ({shape}, {direction}-tailed)")

Skewness of imputation candidates:
  area_sqft     : skew = +0.77  (mildly skewed, right-tailed)
  age_years     : skew = +0.03  (symmetric, right-tailed)
  bathrooms     : skew = +0.41  (symmetric, right-tailed)
  parking_spots : skew = +0.13  (symmetric, right-tailed)


In [33]:
# f-string interpretation tying the skew result to the chosen strategy
mode_furn = df['furnishing_status'].mode()[0] if df['furnishing_status'].notna().any() else 'Unfurnished'

box("output", "Imputation Strategy — Justified Per Column",
    f"<code>area_sqft</code> — skew = <b>{skews['area_sqft']:+.2f}</b>. "
    f"Right-skewed → <b>median</b> (using mean would be pulled by larger flats). Median ≈ "
    f"<b>{df['area_sqft'].median():.0f} sqft</b>.<br><br>"
    f"<code>age_years</code> — skew = <b>{skews['age_years']:+.2f}</b>. "
    f"{'Roughly symmetric' if abs(skews['age_years'])<0.5 else 'Mildly skewed'} → <b>median</b> is a safe default for a numeric column when no domain default applies. "
    f"Median ≈ <b>{df['age_years'].median():.1f} years</b>.<br><br>"
    f"<code>bathrooms</code> — skew = <b>{skews['bathrooms']:+.2f}</b>. "
    f"Discrete count → <b>median</b> (rounds to a sensible whole number). "
    f"Median = <b>{df['bathrooms'].median():.0f}</b>.<br><br>"
    f"<code>parking_spots</code> — <b>0 (domain default)</b>, NOT median. "
    f"In real-estate listings, a blank parking field overwhelmingly means 'no covered parking documented' — "
    f"the median ({df['parking_spots'].median():.0f}) would falsely <i>add</i> parking to listings that don't have it. "
    f"Domain knowledge beats the default statistic here.<br><br>"
    f"<code>furnishing_status</code> — categorical → <b>mode</b> = <b>{mode_furn}</b>. "
    f"There is no median/mean for strings; mode is the only option in the simple toolkit.")

In [34]:
df['area_sqft']        = df['area_sqft'].fillna(df['area_sqft'].median())
df['age_years']        = df['age_years'].fillna(df['age_years'].median())
df['bathrooms']        = df['bathrooms'].fillna(df['bathrooms'].median())
df['parking_spots']    = df['parking_spots'].fillna(0)
df['furnishing_status']= df['furnishing_status'].fillna(df['furnishing_status'].mode()[0])

remaining_missing = df.isna().sum().sum()
print(f"Missing values remaining after imputation: {remaining_missing}")

Missing values remaining after imputation: 0


### Data Quality Dashboard — Verification Asserts

In [35]:
box("definition", "Why a Data Quality Dashboard? — The Cleaning Contract",
    "Cleaning code is just code, and code has bugs. A typo in a filter, a wrong column name in an "
    "imputation, an off-by-one in an outlier rule — any of these can leave your 'cleaned' DataFrame "
    "silently dirty. The model in 11.2 will then train on dirty data, produce great-looking metrics in "
    "development, and fail in production. <b>Asserts are how you catch the bug before it becomes an outage.</b><br><br>"
    "<b>Each assert encodes a contract</b> about what 'clean' means for this dataset:<br>"
    "&nbsp;&nbsp;• 'There must be zero missing values' &nbsp;&nbsp;→&nbsp;&nbsp; <code>assert df.isna().sum().sum() == 0</code><br>"
    "&nbsp;&nbsp;• 'Every BHK must be at least 1' &nbsp;&nbsp;→&nbsp;&nbsp; <code>assert (df['bhk'] > 0).all()</code><br>"
    "&nbsp;&nbsp;• 'No building older than its existence' &nbsp;&nbsp;→&nbsp;&nbsp; <code>assert (df['age_years'] >= 0).all()</code><br><br>"
    "<b>Three rules of engagement.</b><br>"
    "&nbsp;&nbsp;1. Asserts run at the END of cleaning, not during it. They check the contract is satisfied — "
    "if it isn't, fix the cleaning, don't bypass the assert.<br>"
    "&nbsp;&nbsp;2. Each assert message names the specific contract being violated, so the failure tells you "
    "<i>what</i> is wrong, not just <i>that</i> something is wrong.<br>"
    "&nbsp;&nbsp;3. <b>Never comment out a failing assert.</b> If an assert fails, the contract has been violated. "
    "Treat it like a unit-test failure in software engineering — debug the cleaning, do not edit the test.")

In [36]:
# Verification asserts — if ANY assert fails, the notebook stops loudly
assert df.isna().sum().sum() == 0,         "FAIL: Missing values still exist!"
assert (df['bhk'] > 0).all(),              "FAIL: bhk=0 rows remain!"
assert (df['age_years'] >= 0).all(),       "FAIL: Negative age_years remain!"
assert (df['price_lakh'] > 0).all(),       "FAIL: Non-positive price_lakh!"
assert (df['area_sqft'] >= 100).all(),     "FAIL: area_sqft < 100 sqft rows remain!"
assert df['locality'].nunique() >= 25,     "FAIL: Too few unique localities — string cleaning may have missed variants!"
assert 7800 <= len(df) <= 8250,            "FAIL: Row count out of expected range (7800-8250) after cleaning!"

print(f"All cleaning asserts passed.")
print(f"Final clean dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Unique localities: {df['locality'].nunique()}")

All cleaning asserts passed.
Final clean dataset: 8104 rows x 9 columns
Unique localities: 28


In [37]:
box("output", "Cleaning Complete — Summary",
    f"Starting from {before_rows} raw rows:<br>"
    f"- Removed <b>{before_rows - after_rows} duplicate</b> rows<br>"
    f"- Standardised <b>locality</b> and <b>furnishing_status</b> strings (acronym fix-map applied)<br>"
    f"- Converted <b>area_sqft</b> to float (was object/mixed type)<br>"
    f"- Removed <b>{before - after} impossible-value</b> rows (bhk=0, age<0)<br>"
    f"- Winsorized <b>{n_clipped_low + n_clipped_high}</b> price values to the [1st, 99th] percentile band (no rows dropped)<br>"
    f"- Imputed missing: area_sqft / age_years / bathrooms (median), parking_spots (0), furnishing_status (mode)<br>"
    f"<b>Final: {df.shape[0]} clean rows × {df.shape[1]} columns. Zero missing values.</b>")

---

## Section 3.5: Train/Test Split — Do This BEFORE EDA

This is the discipline that separates junior from senior ML engineers.


In [38]:
box("warning", "The Leakage Trap — Why EDA Comes AFTER The Split, Not Before",
    "<b>The trap, in one sentence:</b> if you explore the whole dataset before splitting, your eyes "
    "see patterns that include the test rows — and your feature decisions silently <i>encode those test patterns</i> "
    "into the training pipeline.<br><br>"
    "<b>How it plays out:</b><br>"
    "&nbsp;&nbsp;1. You run EDA on the full DataFrame. You spot some pattern — perhaps a specific feature "
    "combination predicts price unusually well.<br>"
    "&nbsp;&nbsp;2. You engineer a feature that captures that pattern.<br>"
    "&nbsp;&nbsp;3. Because the pattern was visible in the test rows too, the feature 'works' for free "
    "on those rows when you later evaluate.<br>"
    "&nbsp;&nbsp;4. The model looks great in the notebook.<br>"
    "&nbsp;&nbsp;5. In production, on truly unseen listings, the pattern was a coincidence of <i>your</i> "
    "sample — and the model fails.<br><br>"
    "<b>Why this is leakage and not just bias:</b> a model is supposed to be evaluated on data it has "
    "<i>never seen</i>. The moment your feature design uses the test set's structure, the test set is "
    "no longer unseen — it has informed the training pipeline through your judgment.<br><br>"
    "<b>The rule:</b> Split FIRST. Run EDA on <code>train_df</code> only. Fit every transform on "
    "<code>train_df</code>. Apply identical transforms to <code>test_df</code> without recomputing. "
    "<code>test_df</code> stays sealed until 11.2.")

In [39]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,      # 20% held out for honest evaluation
    random_state=42,    # reproducibility
)

# Reset indices for clean downstream indexing
train_df = train_df.reset_index(drop=True).copy()
test_df  = test_df.reset_index(drop=True).copy()

print(f"Train: {train_df.shape[0]} rows ({train_df.shape[0]/len(df)*100:.1f}%)")
print(f"Test:  {test_df.shape[0]} rows ({test_df.shape[0]/len(df)*100:.1f}%)")
print(f"\nTrain price mean:   Rs.{train_df['price_lakh'].mean():.2f}L")
print(f"Test  price mean:   Rs.{test_df['price_lakh'].mean():.2f}L")
print(f"(Similar means = good split, no obvious skew)")

Train: 6483 rows (80.0%)
Test:  1621 rows (20.0%)

Train price mean:   Rs.109.76L
Test  price mean:   Rs.112.27L
(Similar means = good split, no obvious skew)


In [40]:
box("tip", "The Sealed Envelope Rule",
    "Think of <code>test_df</code> as a sealed envelope. You have put it in a drawer. "
    "For the rest of this notebook — and all of 11.2 until final evaluation — "
    "you never open that envelope. Every chart, every statistic, every feature you compute "
    "uses <code>train_df</code> only. When you finally open the envelope in 11.2, "
    "the RMSE you see is an honest estimate of real-world performance.")

---

## Section 4: Exploratory Data Analysis (Train Set Only)

> **Every chart, every statistic, every computation in this section uses `train_df` only.**

We apply the EDA workflow from Session 10.2 at industry pace:
distribution of target → all numeric features → correlation heatmap → bivariate → categorical breakdowns.


### 4.1 Distribution of the Target: price_lakh

**How to read a histogram + KDE overlay:**
- The bars show how many listings fall in each price bin.
- The smooth KDE curve shows the underlying probability density — think of it as a smoothed version of the histogram.
- If the curve is symmetric (like a bell shape) → normal distribution.
- If the curve has a long tail to the RIGHT → right-skewed. The mean will be pulled up by luxury listings.
- **Implication for modelling:** right-skewed targets benefit from a log transform — `np.log1p(price_lakh)` — which we apply in Section 5.


In [41]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=train_df['price_lakh'],
    histnorm='probability density',
    name='price_lakh',
    marker_color='#ED1C24',
    opacity=0.7,
    hovertemplate='Price: Rs.%{x:.1f}L<br>Density: %{y:.4f}<extra></extra>'
))
x_kde, y_kde = kde_line(train_df['price_lakh'])
fig.add_trace(go.Scatter(
    x=x_kde, y=y_kde, mode='lines',
    line=dict(color='black', width=2.5), name='KDE'
))
fig.update_layout(
    title='Price Distribution — Train Set (Rs. Lakhs)',
    xaxis_title='Price (Rs. Lakhs)',
    yaxis_title='Density',
    width=800, height=450,
    template='simple_white'
)
fig.show()

In [42]:
mean_p   = train_df['price_lakh'].mean()
median_p = train_df['price_lakh'].median()
skew_p   = train_df['price_lakh'].skew()

box("output", "Price Distribution: Key Findings",
    f"Mean = Rs.<b>{mean_p:.1f}L</b> | Median = Rs.<b>{median_p:.1f}L</b> | Skew = <b>{skew_p:.2f}</b><br>"
    f"The distribution is <b>right-skewed</b> (mean {'>' if mean_p > median_p else '<='} median, skew {'> 1' if skew_p > 1 else '> 0'}).<br>"
    f"A small number of premium Koramangala/Indiranagar listings pull the mean up.<br>"
    f"<b>Implication:</b> We will apply a <code>log1p</code> transform to <code>price_lakh</code> in Feature Engineering "
    f"to reduce the skew (from {skew_p:.2f} toward 0) before model training.")

### 4.2 Distributions of All Numeric Features

In [43]:
# Multi-panel histogram with KDE overlays — ALL in one cell
num_features = ['area_sqft', 'age_years', 'bhk', 'bathrooms', 'parking_spots']
colors = ['#3498DB', '#2ECC71', '#9B59B6', '#F39C12', '#1ABC9C']

fig = make_subplots(rows=2, cols=3, subplot_titles=num_features + [''],
                    vertical_spacing=0.18, horizontal_spacing=0.10)

for i, (feat, clr) in enumerate(zip(num_features, colors)):
    row = i // 3 + 1
    col = i % 3 + 1
    fig.add_trace(go.Histogram(
        x=train_df[feat], histnorm='probability density',
        marker_color=clr, opacity=0.65, name=feat,
        hovertemplate=f'{feat}: %{{x}}<br>Density: %{{y:.4f}}<extra></extra>'
    ), row=row, col=col)
    try:
        kx, ky = kde_line(train_df[feat])
        fig.add_trace(go.Scatter(
            x=kx, y=ky, mode='lines',
            line=dict(color='black', width=1.8), showlegend=False
        ), row=row, col=col)
    except Exception:
        pass  # skip KDE if column has < 2 unique values

fig.update_layout(
    title_text='Numeric Feature Distributions — Train Set',
    height=550, width=1000,
    showlegend=False,
    plot_bgcolor='white'
)
fig.show()

In [44]:
area_mean  = train_df['area_sqft'].mean()
age_median = train_df['age_years'].median()
bhk_mode   = int(train_df['bhk'].mode()[0])

box("output", "Numeric Feature Distributions: Key Findings",
    f"<b>area_sqft:</b> Mean = {area_mean:.0f} sqft. Right-skewed (large properties pull mean up).<br>"
    f"<b>age_years:</b> Median = {age_median:.1f} years. Fairly uniform distribution (new to older builds).<br>"
    f"<b>bhk:</b> Mode = {bhk_mode} BHK. 2BHK and 3BHK dominate the market.<br>"
    f"<b>bathrooms & parking_spots:</b> Discrete, roughly match bhk distribution as expected.")

### 4.3 Correlation Heatmap

**How to read a correlation heatmap:**
- Each cell shows the Pearson correlation coefficient (r) between two numeric columns.
- **r close to +1** (dark red): strong positive linear relationship.
- **r close to −1** (dark blue): strong negative relationship.
- **r close to 0** (white): no linear relationship.
- The diagonal is always 1.0 (a variable is perfectly correlated with itself).


In [45]:
# Correlation heatmap — ALL traces in one cell
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude listing_id (it's an identifier, not a feature)
numeric_cols = [c for c in numeric_cols if c != 'listing_id']
corr = train_df[numeric_cols].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.columns,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}',
    hovertemplate='%{x} vs %{y}: r = %{z:.3f}<extra></extra>'
))
fig.update_layout(
    title='Pearson Correlation — Numeric Features (Train Set)',
    width=700, height=550,
    template='simple_white'
)
fig.show()

In [46]:
# Find the feature most correlated with price_lakh (excluding price itself)
price_corr = corr['price_lakh'].drop('price_lakh').abs().sort_values(ascending=False)
top_feature = price_corr.index[0]
top_r = corr['price_lakh'][top_feature]
second_feature = price_corr.index[1]
second_r = corr['price_lakh'][second_feature]

box("output", "Correlation Findings",
    f"<b>Strongest predictor of price_lakh:</b> <code>{top_feature}</code> (r = {top_r:.3f})<br>"
    f"<b>Second strongest:</b> <code>{second_feature}</code> (r = {second_r:.3f})<br>"
    f"These two features will be the most influential in our linear regression model in 11.2.<br>"
    f"<b>area_sqft and bhk are also correlated with each other</b> — larger flats have more bedrooms. "
    f"This multicollinearity is worth noting when we interpret model coefficients in 11.2.")

### 4.4 Bivariate: area_sqft vs price_lakh

In [47]:
# Check correlation BEFORE choosing chart type (10.2 rule)
r_area_price = train_df[['area_sqft', 'price_lakh']].corr().iloc[0, 1]
print(f"Pearson r(area_sqft, price_lakh) = {r_area_price:.3f}")
print(f"Decision: {'Scatter plot (|r| >= 0.3)' if abs(r_area_price) >= 0.3 else 'Boxplot by category (|r| < 0.3)'}")

Pearson r(area_sqft, price_lakh) = 0.676
Decision: Scatter plot (|r| >= 0.3)


In [48]:
# area_sqft vs price_lakh scatter (r >= 0.3 expected)
fig = go.Figure(go.Scatter(
    x=train_df['area_sqft'],
    y=train_df['price_lakh'],
    mode='markers',
    marker=dict(color='#ED1C24', opacity=0.5, size=5),
    hovertemplate=(
        'Area: %{x:.0f} sqft<br>Price: Rs.%{y:.1f}L<extra></extra>'
    )
))
fig.update_layout(
    title=f'area_sqft vs price_lakh (Train) — r = {r_area_price:.3f}',
    xaxis_title='Area (sqft)',
    yaxis_title='Price (Rs. Lakhs)',
    width=750, height=430,
    template='simple_white'
)
fig.show()

In [49]:
box("output", "Area vs Price: Bivariate Finding",
    f"Pearson r = <b>{r_area_price:.3f}</b> → {'strong' if abs(r_area_price) >= 0.5 else 'moderate'} positive correlation.<br>"
    f"Larger flats command higher prices — as expected from first principles.<br>"
    f"The scatter shows some vertical spread at each area level, indicating that "
    f"<b>locality and furnishing also drive price</b> beyond area alone.<br>"
    f"Linear Regression will capture the overall slope. The residual spread is what KNN can potentially improve.")

### 4.5 Price by Locality — Which Neighbourhoods Command a Premium?

In [50]:
# Boxplot by locality — sorted by median price
loc_medians = train_df.groupby('locality')['price_lakh'].median().sort_values()
sorted_locs = loc_medians.index.tolist()

fig = go.Figure()
for loc in sorted_locs:
    fig.add_trace(go.Box(
        y=train_df[train_df['locality'] == loc]['price_lakh'],
        name=loc,
        marker_color='#ED1C24',
        hovertemplate=f'{loc}<br>Price: Rs.%{{y:.1f}}L<extra></extra>'
    ))
fig.update_layout(
    title='Price Distribution by Locality (Train) — Sorted by Median',
    yaxis_title='Price (Rs. Lakhs)',
    xaxis_title='Locality',
    width=950, height=500,
    template='simple_white',
    showlegend=False
)
fig.show()

In [51]:
premium_loc  = loc_medians.index[-1]
outskirts_loc = loc_medians.index[0]
premium_med  = loc_medians.iloc[-1]
outskirts_med = loc_medians.iloc[0]

box("output", "Price by Locality: Key Findings",
    f"<b>Highest median:</b> {premium_loc} at Rs.<b>{premium_med:.1f}L</b><br>"
    f"<b>Lowest median:</b> {outskirts_loc} at Rs.<b>{outskirts_med:.1f}L</b><br>"
    f"<b>Premium gap:</b> {premium_loc} listings are ~{premium_med/outskirts_med:.1f}x "
    f"the price of {outskirts_loc} listings for comparable specs.<br>"
    f"This confirms that locality is a strong predictor — our frequency-encoding feature "
    f"(<code>locality_freq</code>) in Section 5 will capture this variation numerically.")

### 4.6 Price by Furnishing Status — Does Furnishing Move the Needle?

In [52]:
# Boxplot of price by furnishing_status — three categories
furn_order = ['Unfurnished', 'Semi-Furnished', 'Furnished']
fig = go.Figure()
for furn in furn_order:
    sub = train_df[train_df['furnishing_status'] == furn]['price_lakh']
    fig.add_trace(go.Box(
        y=sub,
        name=furn,
        marker_color='#ED1C24',
        boxpoints='outliers',
        hovertemplate=f'{furn}<br>Price: Rs.%{{y:.1f}}L<extra></extra>',
    ))
fig.update_layout(
    title='Price Distribution by Furnishing Status (Train)',
    yaxis_title='Price (Rs. Lakhs)',
    xaxis_title='Furnishing',
    width=750, height=420,
    template='simple_white',
    showlegend=False,
)
fig.show()

In [53]:
med_unfurn = train_df[train_df['furnishing_status']=='Unfurnished']['price_lakh'].median()
med_semi   = train_df[train_df['furnishing_status']=='Semi-Furnished']['price_lakh'].median()
med_furn   = train_df[train_df['furnishing_status']=='Furnished']['price_lakh'].median()
premium_pct = (med_furn / med_unfurn - 1) * 100 if med_unfurn else 0

box("output", "Price by Furnishing: Key Findings",
    f"<b>Median by category:</b> Unfurnished = Rs.{med_unfurn:.1f}L | "
    f"Semi-Furnished = Rs.{med_semi:.1f}L | Furnished = Rs.{med_furn:.1f}L<br>"
    f"<b>Furnished premium:</b> Furnished listings sit roughly <b>{premium_pct:+.1f}%</b> above Unfurnished. "
    f"Some of this gap is the furniture itself; some is selection bias (Furnished listings cluster in premium localities). "
    f"Either way, <code>furnishing_status</code> carries signal — we will encode it (one-hot or frequency) before modelling in 11.2.")

### 4.7 Price by BHK — Does Bedroom Count Drive Price?

In [54]:
# Boxplot of price by bhk — discrete numeric, treated as categorical here
bhk_order = sorted(train_df['bhk'].unique())
fig = go.Figure()
for b in bhk_order:
    sub = train_df[train_df['bhk'] == b]['price_lakh']
    fig.add_trace(go.Box(
        y=sub,
        name=f'{int(b)} BHK',
        marker_color='#ED1C24',
        boxpoints='outliers',
        hovertemplate=f'{int(b)} BHK<br>Price: Rs.%{{y:.1f}}L<extra></extra>',
    ))
fig.update_layout(
    title='Price Distribution by BHK (Train)',
    yaxis_title='Price (Rs. Lakhs)',
    xaxis_title='Bedrooms-Hall-Kitchen (BHK)',
    width=750, height=420,
    template='simple_white',
    showlegend=False,
)
fig.show()

In [55]:
bhk_medians = train_df.groupby('bhk')['price_lakh'].median().to_dict()
med_lines = ' | '.join(f"{int(k)}BHK = Rs.{v:.1f}L" for k, v in sorted(bhk_medians.items()))
ratio = (bhk_medians.get(3, 0) / bhk_medians.get(1, 1)) if bhk_medians.get(1) else float('nan')

box("output", "Price by BHK: Key Findings",
    f"<b>Median price by BHK:</b> {med_lines}.<br>"
    f"<b>3-BHK vs 1-BHK ratio:</b> ~{ratio:.1f}x — bedroom count is a strong driver, "
    f"largely because area scales with BHK (which we already saw in the area-vs-price scatter).<br>"
    f"<b>Implication:</b> <code>bhk</code> stays as a model feature; we also engineer <code>sqft_per_bhk</code> "
    f"in Section 5 to capture <i>density</i> separately from <i>raw size</i> — a cramped 3BHK is priced "
    f"differently from a spacious 3BHK at the same total area.")

---

## Section 5: Feature Engineering — Fit on Train, Apply to Test

**The golden rule:** Every transform must be computed from `train_df` only. Then apply the *identical* transform to `test_df`. Never compute statistics on test data.

We engineer four features:
1. `log_price` — log transform of the right-skewed target
2. `price_per_sqft_eda_only` — target-derived ratio (LEAKAGE WARNING — EDA only, not a model feature)
3. `locality_freq` — frequency encoding of locality (fit on train, mapped to test)
4. `sqft_per_bhk` — size efficiency ratio


### Feature 1: log_price — Taming the Right Skew

In [56]:
# Apply to BOTH train and test — no train statistics involved, so this is safe
train_df['log_price'] = np.log1p(train_df['price_lakh'])
test_df['log_price']  = np.log1p(test_df['price_lakh'])

original_skew = train_df['price_lakh'].skew()
log_skew      = train_df['log_price'].skew()
print(f"price_lakh skew:  {original_skew:.3f}")
print(f"log_price skew:   {log_skew:.3f}")
print(f"Skew reduction:   {original_skew - log_skew:.3f} (closer to 0 = more symmetric)")

price_lakh skew:  1.430
log_price skew:   -0.099
Skew reduction:   1.529 (closer to 0 = more symmetric)


**Visual confirmation that the log transform fixed the skew.** Compare this histogram + KDE of `log_price` to the original `price_lakh` histogram from Section 4.1 — the right tail should be gone and the curve should be roughly symmetric (close to bell-shaped).

In [57]:
# Histogram + KDE of log_price (train) — visual proof the log transform worked
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=train_df['log_price'],
    histnorm='probability density',
    name='log_price',
    marker_color='#ED1C24',
    opacity=0.7,
    hovertemplate='log_price: %{x:.2f}<br>Density: %{y:.3f}<extra></extra>',
))
x_kde, y_kde = kde_line(train_df['log_price'])
fig.add_trace(go.Scatter(
    x=x_kde, y=y_kde, mode='lines',
    line=dict(color='black', width=2.5), name='KDE',
))
fig.update_layout(
    title='Distribution of log_price (Train) — After log1p Transform',
    xaxis_title='log_price = log(1 + price_lakh)',
    yaxis_title='Density',
    width=800, height=420,
    template='simple_white',
)
fig.show()

In [58]:
log_mean   = train_df['log_price'].mean()
log_median = train_df['log_price'].median()

box("output", "Log-Price Distribution: Before vs After",
    f"<b>Before transform (price_lakh):</b> skew = <b>{original_skew:+.2f}</b> — strongly right-tailed.<br>"
    f"<b>After transform (log_price):</b> skew = <b>{log_skew:+.2f}</b> — "
    f"{'roughly symmetric' if abs(log_skew) < 0.5 else 'mildly skewed but much closer to symmetric'}.<br>"
    f"<b>Mean ≈ median:</b> log_price mean = {log_mean:.2f} | log_price median = {log_median:.2f}. "
    f"The two converge as the distribution becomes symmetric — exactly what we wanted before "
    f"feeding this into a Linear Regression in 11.2 (which assumes symmetric residuals).")

### Feature 2: price_per_sqft — EDA Only (Target Leakage Warning)

In [59]:
box("warning", "Target Leakage: price_per_sqft Cannot Be a Model Feature",
    "<code>price_per_sqft = (price_lakh * 100000) / area_sqft</code><br>"
    "This ratio is derived <b>directly from the target variable</b> (<code>price_lakh</code>). "
    "If we include it as a model feature, the model is essentially using the answer to predict itself "
    "— evaluation metrics will look perfect, but the feature will be unavailable at prediction time "
    "(because you do not know the price of a listing you are trying to estimate).<br><br>"
    "<b>We compute it for EDA insight only. It does NOT go into any model feature set.</b>")

In [60]:
# EDA-only feature: price per sqft (train only — do NOT add to test_df)
train_df['price_per_sqft_eda_only'] = (
    train_df['price_lakh'] * 100_000 / train_df['area_sqft']
).round(0)

mean_ppsf = train_df['price_per_sqft_eda_only'].mean()
print(f"Mean price per sqft in train: Rs.{mean_ppsf:,.0f} per sqft")
print(f"Note: this column is on train_df only — NOT a model feature.")

Mean price per sqft in train: Rs.8,010 per sqft
Note: this column is on train_df only — NOT a model feature.


### Feature 3: locality_freq — Frequency Encoding (Fit on Train, Map to Test)

In [61]:
# Compute frequency map from TRAIN only
freq_map = train_df['locality'].value_counts(normalize=True).to_dict()

print("Locality frequency map (from train):")
for loc, freq in sorted(freq_map.items(), key=lambda x: -x[1]):
    print(f"  {loc:<20}: {freq:.4f}")

# Map to BOTH train and test (.fillna(0) handles any test locality not seen in train)
train_df['locality_freq'] = train_df['locality'].map(freq_map)
test_df['locality_freq']  = test_df['locality'].map(freq_map).fillna(0)

print(f"\nTrain NaN in locality_freq: {train_df['locality_freq'].isna().sum()}")
print(f"Test  NaN in locality_freq: {test_df['locality_freq'].isna().sum()}")

Locality frequency map (from train):
  Whitefield          : 0.0751
  Sarjapur Road       : 0.0609
  Koramangala         : 0.0549
  HSR Layout          : 0.0529
  Indiranagar         : 0.0512
  Jayanagar           : 0.0481
  Electronic City     : 0.0470
  Bellandur           : 0.0455
  Marathahalli        : 0.0446
  Banashankari        : 0.0404
  Bannerghatta Road   : 0.0389
  Yelahanka           : 0.0366
  Hebbal              : 0.0356
  BTM Layout          : 0.0356
  JP Nagar            : 0.0342
  KR Puram            : 0.0305
  Kanakapura Road     : 0.0302
  Thanisandra         : 0.0295
  Hennur              : 0.0282
  Hoodi               : 0.0276
  Domlur              : 0.0236
  CV Raman Nagar      : 0.0228
  Mysore Road         : 0.0219
  Bommasandra         : 0.0213
  Begur               : 0.0190
  Devanahalli         : 0.0168
  Attibele            : 0.0162
  Sadashivnagar       : 0.0106

Train NaN in locality_freq: 0
Test  NaN in locality_freq: 0


### Feature 4: sqft_per_bhk — Size Efficiency Ratio

In [62]:
# Simple ratio — no train statistics involved
train_df['sqft_per_bhk'] = train_df['area_sqft'] / train_df['bhk']
test_df['sqft_per_bhk']  = test_df['area_sqft'] / test_df['bhk']

mean_spb = train_df['sqft_per_bhk'].mean()
print(f"Mean sqft per BHK in train: {mean_spb:.1f} sqft")
print(f"This captures space efficiency — a 1,200 sqft 3BHK is 400 sqft/BHK (cramped vs spacious for the price).")

Mean sqft per BHK in train: 545.1 sqft
This captures space efficiency — a 1,200 sqft 3BHK is 400 sqft/BHK (cramped vs spacious for the price).


### Verification: All Engineered Features in Both Train and Test

In [63]:
new_features = ['log_price', 'locality_freq', 'sqft_per_bhk']

for f in new_features:
    assert f in train_df.columns, f"FAIL: {f} missing from train_df!"
    assert f in test_df.columns,  f"FAIL: {f} missing from test_df!"
    assert train_df[f].notna().all(), f"FAIL: {f} has NaN in train_df!"
    assert test_df[f].notna().all(),  f"FAIL: {f} has NaN in test_df!"

print(f"All {len(new_features)} engineered features verified in train_df and test_df.")
print(f"\ntrain_df columns: {train_df.shape[1]}")
print(f"test_df  columns: {test_df.shape[1]}")
print(f"Note: test_df has 1 fewer column (price_per_sqft_eda_only is train-only, not a model feature)")

All 3 engineered features verified in train_df and test_df.

train_df columns: 13
test_df  columns: 12
Note: test_df has 1 fewer column (price_per_sqft_eda_only is train-only, not a model feature)


In [64]:
box("output", "Feature Engineering Summary",
    "<b>4 features engineered:</b><br>"
    "1. <code>log_price</code> — log1p transform of target. "
    f"Skew reduced from {original_skew:.2f} to {log_skew:.2f}. Applied to both train and test.<br>"
    "2. <code>price_per_sqft_eda_only</code> — target-derived ratio. "
    "EDA only. NOT a model feature (leakage risk).<br>"
    "3. <code>locality_freq</code> — frequency encoding. "
    "Fit on train value_counts, mapped to test with .fillna(0).<br>"
    "4. <code>sqft_per_bhk</code> — area per bedroom. "
    "Captures space efficiency. Applied to both train and test.")

---

## Section 6: Baseline Model & Project Save


In [65]:
box("industry", "Why Every Project Needs a Baseline",
    "At McKinsey & Company's data science teams, the first deliverable on any ML project "
    "is not a model — it is a <b>baseline benchmark document</b>. It answers: "
    "'What does the current state-of-the-art non-ML solution achieve?' "
    "In the absence of an existing system, the baseline is the mean prediction. "
    "<b>If your model cannot beat this, you do not have a model — you have an expensive calculator.</b> "
    "Every ML engineer should internalise this: your model's value is measured against the baseline, "
    "not in absolute terms.")

### Computing the Baseline RMSE

In [66]:
# Baseline: predict mean train price for every test row
mean_price_train = train_df['price_lakh'].mean()
baseline_pred    = np.full(len(test_df), mean_price_train)
baseline_rmse    = np.sqrt(mean_squared_error(test_df['price_lakh'], baseline_pred))
target_rmse      = baseline_rmse * 0.70  # 30% improvement

print(f"Mean train price :  Rs.{mean_price_train:.2f}L")
print(f"Baseline RMSE    :  Rs.{baseline_rmse:.2f}L")
print()
print(f"Any model in 11.2 MUST beat Rs.{baseline_rmse:.2f}L RMSE.")
print(f"Success threshold:  Rs.{target_rmse:.2f}L RMSE (30% improvement over baseline)")

Mean train price :  Rs.109.76L
Baseline RMSE    :  Rs.63.31L

Any model in 11.2 MUST beat Rs.63.31L RMSE.
Success threshold:  Rs.44.32L RMSE (30% improvement over baseline)


In [67]:
box("output", "Baseline Benchmark Established",
    f"<b>Mean train price:</b> Rs.{mean_price_train:.2f}L<br>"
    f"<b>Baseline RMSE:</b> Rs.<b>{baseline_rmse:.2f}L</b> "
    f"(predicting Rs.{mean_price_train:.2f}L for every test listing)<br>"
    f"<b>Success threshold:</b> Rs.{target_rmse:.2f}L RMSE (30% better than baseline)<br><br>"
    f"Interpretation: The baseline model is off by an average of Rs.{baseline_rmse:.2f}L on each listing. "
    f"A trained Linear Regression or KNN model should do significantly better by using listing features.")

### Saving the Project Files

In [68]:
# Save train and test as separate CSVs — the inputs to Session 11.2
train_df.to_csv('bengaluru_train.csv', index=False)
test_df.to_csv('bengaluru_test.csv', index=False)

print("Saved project files:")
print(f"  bengaluru_train.csv  ->  {train_df.shape[0]} rows x {train_df.shape[1]} columns")
print(f"  bengaluru_test.csv   ->  {test_df.shape[0]} rows x {test_df.shape[1]} columns")
print()
print(f"Baseline RMSE to beat in 11.2: Rs.{baseline_rmse:.2f}L")
print(f"Success threshold            : Rs.{target_rmse:.2f}L (30% improvement)")

Saved project files:
  bengaluru_train.csv  ->  6483 rows x 13 columns
  bengaluru_test.csv   ->  1621 rows x 12 columns

Baseline RMSE to beat in 11.2: Rs.63.31L
Success threshold            : Rs.44.32L (30% improvement)


In [69]:
box("tip", "These Are Your 11.2 Inputs",
    f"In Session 11.2, open <code>bengaluru_train.csv</code> ({train_df.shape[0]} rows) and "
    f"<code>bengaluru_test.csv</code> ({test_df.shape[0]} rows). "
    f"No re-cleaning needed. The features <code>log_price</code>, <code>locality_freq</code>, "
    f"<code>sqft_per_bhk</code> are already in both files. "
    f"You will train Linear Regression and KNN Regressor on <code>train_df</code>, "
    f"then evaluate on <code>test_df</code> and compare against the "
    f"Rs.{baseline_rmse:.2f}L baseline RMSE established here.")

---

## Common Pitfalls — What Goes Wrong on Day 1 Projects

| Pitfall | The Fix |
|---------|----------|
| Imputing missing values BEFORE removing impossibles | Follow the order: Duplicates → Strings → Types → Impossibles → Outliers → Missing (last) |
| EDA on the full dataset before splitting | Split first. EDA on `train_df` only. The test set is a sealed envelope. |
| Frequency-encoding using the full dataset's value counts | Build `freq_map` from `train_df.value_counts()` only, then `.map(freq_map).fillna(0)` on test |
| Using `price_per_sqft` as a model feature | It is derived from the target → leakage. Use only for EDA. |
| Skipping the baseline | Without a baseline, you cannot tell if your model has any signal at all |
| Hardcoded interpretation text in notebooks | Always use f-strings on computed statistics — hardcoded numbers go stale when data changes |


## Key Takeaways

1. **Problem framing first.** Convert a vague business brief into target + success metric + baseline + data dictionary BEFORE writing any modelling code.
2. **6-step cleaning order.** Duplicates → Strings → Types → Impossibles → Outliers → Missing (always last). Order matters because each step's statistics depend on prior steps being clean.
3. **Train/test split BEFORE EDA.** Looking at test data while exploring leaks information into your feature engineering decisions.
4. **EDA on train only.** Distributions with KDE, correlation heatmap, categorical breakdowns — all on `train_df`.
5. **Feature engineering: fit on train, apply to test.** Frequency encodings are built from `train_df.value_counts()` only, then mapped to test with `.fillna(0)`.
6. **Avoid target leakage.** Features derived from the target (`price_per_sqft` from `price_lakh`) cannot go into any model — only into EDA.
7. **Compute a baseline.** Predict the mean training price for every test row. Any real model must beat this RMSE.
8. **Save train + test separately.** Two CSVs go into 11.2 — not a merged dataframe.
9. **Verification asserts at every milestone.** End of cleaning. End of feature engineering. Fail loudly on data quality violations.
10. **Day 1 of a real ML project looks exactly like this.** This workflow repeats at every company, every team, every dataset.


## What's Next: Session 11.2 — Model Training & Evaluation

```
Today's outputs (saved files)                  11.2 inputs
────────────────────────────────────────       ──────────────────────────────────
bengaluru_train.csv (cleaned + FE)        -->  Train 3 models on this
bengaluru_test.csv  (cleaned + FE)        -->  Honest evaluation here
Baseline RMSE                             -->  The floor every model must beat
Problem statement (target, metric)        -->  Success criteria for the project
Data dictionary                           -->  Documentation handoff to 11.2
```

In Session 11.2 you will train **Linear Regression** and **KNN Regressor** on `bengaluru_train.csv`, evaluate on `bengaluru_test.csv`, compare RMSE against the baseline (the success threshold is to **beat the baseline by at least 30%**), plot residual distributions, and analyse errors by locality.


---

Vishlesan i-Hub IIT Patna × Masai School